# Projeto ETL #

Importando as bibliotecas necessárias.

In [110]:
import pandas as pd
import unicodedata
from thefuzz import fuzz          
from itertools import combinations 


Recebendo a base de dados.

In [111]:
df = pd.read_csv('../Planilhas/imoveis_unificado.csv', sep=';', encoding='utf-8', skiprows=1)
df.head()

,logradouro,numero,complemento,valor_avaliacao,bairro,cidade,uf,ano_construcao,area_terreno,area_construida,...,tipo_ocupacao,data_transacao,estado_conservacao,tipo_imovel,sfh,cod_logradouro,latitude,longitude,ano,Distrito
0,av norte miguel arraes de alencar,3071,NaN,1068562.63,Encruzilhada,Recife,PE,1997,"438,0","511,0",...,COMERCIAL COM LIXO ORGANICO,2023-12-21,Regular,Galpão,0.00,46540,-8.034273,-34.896337,2023,1.0
1,av norte miguel arraes de alencar,3029,NaN,1500000.00,Encruzilhada,Recife,PE,1957,"779,33","582,44",...,COMERCIAL SEM LIXO ORGANICO,2023-11-17,Regular,Casa,0.00,46540,-8.034435,-34.896335,2023,1.0
2,rua belmiro correa,133,apto 1,110000.00,Encruzilhada,Recife,PE,1970,"562,05","121,0",...,RESIDENCIAL,2023-09-22,Bom,Apartamento,0.00,10715,-8.035013,-34.895903,2023,1.0
3,rua belmiro correa,133,apto 1,110000.00,Encruzilhada,Recife,PE,1970,"562,05","121,0",...,RESIDENCIAL,2023-09-26,Bom,Apartamento,0.00,10715,-8.035013,-34.895903,2023,1.0
4,rua belmiro correa,133,apto 2,110000.00,Encruzilhada,Recife,PE,1970,"562,05","81,0",...,RESIDENCIAL,2023-09-22,Bom,Apartamento,0.00,10715,-8.035013,-34.895903,2023,1.0


In [112]:
df.columns.tolist()

['logradouro',
 'numero',
 'complemento',
 'valor_avaliacao',
 'bairro',
 'cidade',
 'uf',
 'ano_construcao',
 'area_terreno',
 'area_construida',
 'fracao_ideal',
 'padrao_acabamento',
 'tipo_construcao',
 'tipo_ocupacao',
 'data_transacao',
 'estado_conservacao',
 'tipo_imovel',
 'sfh',
 'cod_logradouro',
 'latitude',
 'longitude',
 'ano',
 'Distrito']

# Tratamento das Colunas #


### **Tratamento da coluna 'Logradouro'** ###


Criando uma variável para investigar apenas a coluna desejada.

In [113]:
df_tratar = df['logradouro']

Verificamos o tipo de dado e a quantidade de entradas não nulas.

In [114]:
#info() exibe o tipo de dado e a contagem de valores não nulos
#Esperamos tipo 'object' (texto) — logradouro é uma coluna de texto livre
df_tratar.info()

<class 'pandas.Series'>
RangeIndex: 44231 entries, 0 to 44230
Series name: logradouro
Non-Null Count  Dtype
--------------  -----
44231 non-null  str  
dtypes: str(1)
memory usage: 345.7 KB


Verificamos a frequência de cada logradouro para entender a distribuição dos dados.

In [115]:
#Conta a frequência de cada logradouro para entender a distribuição
#Alta cardinalidade é esperada (cada rua é um valor distinto)
#Observar se há valores claramente inválidos no topo ou na cauda da lista
df_tratar.value_counts()

logradouro
av boa viagem                  1178
4tr pereira barreto             673
rua dos navegantes              621
av da recuperacao               522
av republica do libano          512
                               ... 
rua francisco de assis            1
rua sargento melo junior          1
rua baronesa de palmares          1
rua jean mermoz                   1
rua capitao barroso pereira       1
Name: count, Length: 2237, dtype: int64

Obtemos um resumo estatístico da coluna.

In [116]:
#describe() para colunas de texto mostra: total, únicos, valor mais frequente e sua contagem
#O campo 'unique' indica a cardinalidade real da coluna
df_tratar.describe()

count             44231
unique             2237
top       av boa viagem
freq               1178
Name: logradouro, dtype: object

Calculamos o percentual de valores nulos na coluna.

In [117]:
#Calcula o percentual de nulos na coluna logradouro
#Resultado 0.0 confirma que todos os registros possuem logradouro preenchido
df_tratar.isnull().mean() * 100

np.float64(0.0)

Listamos todos os valores distintos para identificar possíveis inconsistências.

In [118]:
#Lista todos os valores distintos para inspeção visual
#Permite identificar encoding corrompido (ex: 'josÃ©'), abreviações (ex: 'eng sampaio')
#e outros padrões inconsistentes que passariam despercebidos em análises agregadas
df_tratar.unique()

<StringArray>
[  'av norte miguel arraes de alencar',                  'rua belmiro correa',
                    'rua caio pereira',                    'av santos dumont',
               'rua doutor jose maria',                  'rua andre reboucas',
              'rua engenheiro sampaio',                     'rua eng sampaio',
                  'rua regueira costa',            'rua general abreu e lima',
 ...
                'rua escola de sagres', 'av centenario alberto santos dumont',
            'rua adauto carneiro leal',  'rua coronel sergio henrique cardim',
              'rua francisco de assis',            'rua sargento melo junior',
            'rua baronesa de palmares',                 'rua conde vila flor',
                     'rua jean mermoz',         'rua capitao barroso pereira']
Length: 2237, dtype: str

Utilizamos a biblioteca thefuzz para comparar logradouros com similaridade maior que 90%. Filtramos apenas os que possuem o mesmo tipo de via e tamanho parecido, evitando falsos positivos. Isso permite detectar inconsistências que passariam despercebidas em uma inspeção manual.

In [119]:

logradouros = df['logradouro'].unique().tolist()

suspeitos = []
for a, b in combinations(logradouros, 2):
    #Filtra: apenas compara logradouros do mesmo tipo de via (rua com rua, av com av...)
    #Isso evita falsos positivos como comparar 'rua X' com 'av Y'
    if a.split()[0] != b.split()[0]:
        continue

    #Filtra: descarta pares com diferença de tamanho maior que 30%
    #Elimina comparações como 'rua eng' vs 'rua engenheiro abreu e lima' (muito diferentes em tamanho)
    if abs(len(a) - len(b)) / max(len(a), len(b)) > 0.30:
        continue

    #token_sort_ratio ordena as palavras antes de comparar, tornando a similaridade mais robusta a diferenças de ordem entre as palavras
    score = fuzz.token_sort_ratio(a, b)

    #Apenas pares com similaridade >= 90% são considerados suspeitos de serem o mesmo logradouro
    if score >= 90:
        suspeitos.append((score, a, b))

suspeitos.sort(reverse=True)
for score, a, b in suspeitos:
    print(f"{score}% | {a} | {b}")
print(len(suspeitos))

100% | rua sant'anna | rua sant anna
100% | av sul gov cid sampaio | av sul gov. cid sampaio
98% | rua jose domingues da silva | rua josé domingues da silva
98% | rua joao fernandes vieira | rua joão fernandes vieira
98% | rua doutor jose maria | rua doutor josé maria
97% | rua teles junior | rua teles júnior
97% | rua simao mendes | rua simão mendes
97% | rua neto de mendonca | rua neto de mendonça
97% | rua mário domingues | rua mario domingues
97% | rua martins junior | rua martins júnior
97% | rua mamede simoes | rua mamede simões
97% | rua do príncipe | rua do principe
97% | rua do hospicio | rua do hospício
97% | prc da independencia | prc da independência
97% | av joao de barros | av joão de barros
96% | rua maest fernando borges | rua maestro fernando borges
96% | rua jurupatan | rua jurupata
96% | rua da uniao | rua da união
96% | rua da gloria | rua da glória
96% | rua caracatuba | rua aracatuba
95% | rua jundia | rua jundiai
95% | rua doutor jose maria | rua doutor jose mari

O fuzzy matching com filtro de tamanho não captura pares onde a abreviação é muito menor que a forma completa, como 'eng' versus 'engenheiro'. Esta célula complementa a análise buscando diretamente pelos padrões conhecidos de abreviação.

In [120]:
#O fuzzy matching com filtro de tamanho (30%) não captura pares onde a abreviação é muito menor que a forma completa (ex: 'eng' vs 'engenheiro', diferença de ~55%)
# Esta célula complementa a análise buscando diretamente pelos padrões conhecidos de abreviação

abrevs = {'eng': 'engenheiro', 'dr': 'doutor', 'gen': 'general',
          'cel': 'coronel', 'ten': 'tenente', 'prof': 'professor',
          'pe': 'padre', 'cap': 'capitao', 'gov': 'governador',
          'pref': 'prefeito', 'sen': 'senador', 'sgto': 'sargento',
          'profa': 'professora', 'maest': 'maestro'}

for abrev, completo in abrevs.items():
    #Busca logradouros que contêm a abreviação como palavra isolada (\b = word boundary)
    # regex=True → interpreta '\b' como limite de palavra, não texto literal, sem isso, '\beng\b' seria buscado literalmente e nunca encontraria nada
    com_abrev    = df[df['logradouro'].str.contains(rf'\b{abrev}\b', regex=True)]['logradouro'].unique()
    #Busca logradouros que contêm a forma completa correspondente
    com_completo = df[df['logradouro'].str.contains(rf'\b{completo}\b', regex=True)]['logradouro'].unique()

    # Só exibe se ambas as formas existem simultaneamente no dataset (inconsistência confirmada)
    if len(com_abrev) > 0 and len(com_completo) > 0:
        print(f"\n{abrev} ({len(com_abrev)}) / {completo} ({len(com_completo)}):")
        for a in com_abrev[:5]:
            print(f"  ABR:  {a}")
        for c in com_completo[:5]:
            print(f"  COMP: {c}")


eng (6) / engenheiro (24):
  ABR:  rua eng sampaio
  ABR:  av eng agamenon de magalhaes melo
  ABR:  rua eng leonardo arcoverde
  ABR:  rua eng henrique marques lins
  ABR:  rua eng jose brandao cavalcante
  COMP: rua engenheiro sampaio
  COMP: rua engenheiro ubaldo gomes de matos
  COMP: av engenheiro agamenon de magalhaes melo
  COMP: rua engenheiro nestor moreira reis
  COMP: rua engenheiro domber

dr (79) / doutor (21):
  ABR:  rua dr vicente meira
  ABR:  av dr malaquias
  ABR:  rua dr geraldo de andrade
  ABR:  rua dr joaquim de arruda falcao
  ABR:  rua dr fernando allain
  COMP: rua doutor jose maria
  COMP: rua doutor joaquim portela
  COMP: rua doutor eduardo wanderley
  COMP: rua doutor carlos mavignier
  COMP: rua doutor joao lacerda

gen (5) / general (21):
  ABR:  rua gen joaquim inacio
  ABR:  rua gen vargas
  ABR:  rua gen polidoro
  ABR:  rua gen americano freire
  ABR:  rua gen salgado
  COMP: rua general abreu e lima
  COMP: rua general artur oscar
  COMP: rua gener

Podemos verificar que a análise revelou três tipos de inconsistências na coluna logradouro. O primeiro é o encoding corrompido: logradouros com caracteres como Ã©, Ã£ e Ã§ são versões mal decodificadas de letras acentuadas, pois partes do arquivo foram gravadas em UTF-8 mas lidas como latin-1. O segundo são abreviações inconsistentes, onde o mesmo logradouro aparece com formas abreviadas e por extenso, como 'eng' e 'engenheiro', 'dr' e 'doutor', entre outros. O terceiro são pontuações inconsistentes, como apóstrofos em nomes como 'sant'anna'. Os pares identificados como ruas diferentes, por exemplo 'rua jose maria' e 'rua jose mariano', não foram tratados pois são logradouros distintos.

Assim, aplicamos as correções em sequência. Primeiro removemos a linha de cabeçalho duplicada. Em seguida corrigimos o encoding corrompido, removemos os acentos para unificar versões com e sem acento, expandimos as abreviações para a forma completa e por fim removemos pontuações inconsistentes como apóstrofos e pontos.

In [121]:

# Função para corrigir strings com encoding corrompido
# Ocorre quando caracteres UTF-8 foram gravados mas lidos como latin-1,
# gerando sequências como 'Ã©' no lugar de 'é', 'Ã£' no lugar de 'ã'
def corrigir_encoding(texto):
    try:
        return texto.encode('latin-1').decode('utf-8')
    except:
        return texto  # se não conseguir corrigir, mantém o valor original

#Função para remover acentuação de uma string
# Após corrigir o encoding, alguns logradouros ficam com acento e outros sem
#(ex: 'rua josé maria' e 'rua jose maria'). Remover acentos unifica ambos
def remover_acentos(texto):
    return unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('ascii')

#Primeiro: Remove a linha de cabeçalho duplicada
# Ao unir os CSVs, o header de um arquivo foi incluído como dado
df = df[df['logradouro'] != 'logradouro']

#Segundo: Corrige encoding corrompido (ex: 'josÃ©' → 'josé')
df['logradouro'] = df['logradouro'].apply(corrigir_encoding)

#Terceiro: Remove acentos para unificar versões com e sem acento
# Ex: 'rua josé domingues' e 'rua jose domingues' → ambos viram 'rua jose domingues'
df['logradouro'] = df['logradouro'].apply(remover_acentos)

#Quarto: Expande abreviações para a forma completa
# Lista de tuplas (padrão regex, forma completa)
# \b = word boundary, garante que só substitui a palavra isolada (não parte de outra palavra)
abrevs = [
    (r'\bgov\.', 'governador'),   # gov. com ponto (antes dos sem ponto, pois regex é guloso)
    (r'\beng\b', 'engenheiro'),
    (r'\bdr\b',  'doutor'),
    (r'\bgen\b', 'general'),
    (r'\bcel\b', 'coronel'),
    (r'\bten\b', 'tenente'),
    (r'\bprofa\b', 'professora'), # profa antes de prof para não substituir parcialmente
    (r'\bprof\b', 'professor'),
    (r'\bpe\b',  'padre'),
    (r'\bcap\b', 'capitao'),
    (r'\bgov\b', 'governador'),
    (r'\bpref\b','prefeito'),
    (r'\bsen\b', 'senador'),
    (r'\bsgto\b','sargento'),
    (r'\bmaest\b','maestro'),
]
for padrao, completo in abrevs:
    df['logradouro'] = df['logradouro'].str.replace(padrao, completo, regex=True)

# PASSO 5: Remove pontuação inconsistente e normaliza espaços múltiplos
# Ex: "sant'anna" → "santanna" | "gov." → já tratado acima | "  " → " "
df['logradouro'] = df['logradouro'].str.replace("'", '', regex=False)
df['logradouro'] = df['logradouro'].str.replace(r'\.', '', regex=True)
df['logradouro'] = df['logradouro'].str.strip()

print(f"Únicos antes do tratamento: 2237")
print(f"Únicos após tratamento: {df['logradouro'].nunique()}")
df['logradouro'].value_counts().head(10)

Únicos antes do tratamento: 2237
Únicos após tratamento: 2184


logradouro
av boa viagem                     1178
4tr pereira barreto                673
rua dos navegantes                 621
av da recuperacao                  522
av republica do libano             512
rua amalia bernardino de souza     499
rua visconde de jequitinhonha      489
av conselheiro aguiar              445
av hildebrando de vasconcelos      417
rua emiliano braga                 406
Name: count, dtype: int64

# Tratamento da Coluna Complemento
Obs: coluna com maior quantidade de valores unicos

Observações iniciais dos dados

In [122]:
df['complemento'].info()
print(f"Total de nulos: {df['complemento'].isna().sum()}")
print(f"Total de linhas: {len(df)}")

<class 'pandas.Series'>
RangeIndex: 44231 entries, 0 to 44230
Series name: complemento
Non-Null Count  Dtype
--------------  -----
40082 non-null  str  
dtypes: str(1)
memory usage: 345.7 KB
Total de nulos: 4149
Total de linhas: 44231


In [123]:
print(df['complemento'].value_counts(dropna=False).head(20))


complemento
NaN                  4149
apto 0101              53
apto 2                 49
apto 0102              46
apto 0201              45
apto 1                 43
apto 0202              32
apto 0102 bloco a      28
apto 302 bloco b       27
apto 302 bloco a       27
apto 203 bloco a       27
apto 4                 26
apto 102 bloco a       26
apto 401 bloco a       25
apto 501 bloco b       24
apto 804 bloco a       24
apto 303 bloco a       24
apto 402 bloco a       24
apto 601 bloco a       24
apto 502 bloco a       24
Name: count, dtype: int64


In [124]:
df['complemento'].head(20)

0                                NaN
1                                NaN
2                             apto 1
3                             apto 1
4                             apto 2
5                             apto 3
6                             apto 4
7                          apto 0005
8                                NaN
9                                NaN
10       apto 203 edf sainte juliana
11       apto 201 edf sainte juliana
12      apto 1501 edf sainte juliana
13                               NaN
14    apto 902 edf essenza rosarinho
15       apto 2101 edf. jardim monet
16      apto 1401 edf praia de ceres
17           apto 201 edf maria rosa
18           apto 404 edf maria rosa
19          apto 1304 edf maria rosa
Name: complemento, dtype: str

Aqui já foi percebido uma falta de padronização nos complementos, alguns com numeração (0101) e (101), outros até com nome do edifício

In [125]:
# Ver todos os padrões com "edf" para entender as variações
mask = df['complemento'].str.contains('edf', na=False)
print(df[mask]['complemento'].value_counts().head(30))

complemento
apto 401 edf ocean breeze                       9
apto 0601 edf maria gabriela                    8
apto 0802 edf saint paul                        7
sala 2207 edf. torreão executive plaza          7
apto 1009 edf. tolive one                       5
apto 102 edf flor de macambira                  5
apto 0101 edf bom pr                            5
apto 2203 edf praça dos mognos                  5
apto 305 edf portal da varzea                   5
apto 0701 edf n sra do amparo                   5
sala 0609 edf unicenter emp                     5
apto 1601 edf vivres village                    5
apto 0501 edf hockenheim                        5
apto 204 edf jardins madalena                   5
apto 1507 edf wimbledon boa viagem              5
apto 1102 edf maria eugenia - bloco c           5
apto 1402 edf rita de cassia - bloco a          5
apto 1101 edf rita de cassia - bloco a          5
apto 0202 edf baía helsink                      5
sala 0009 edf canarias                

In [126]:
#Buscar variações com apt e apto
mask = df['complemento'].str.contains('apt\.', na=False, case=False)
print(df[mask]['complemento'].value_counts().head(20))
print(f"\nTotal: {mask.sum()}")

Series([], Name: count, dtype: int64)

Total: 0


<>:2: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
C:\Users\USER\AppData\Local\Temp\ipykernel_11948\1136421536.py:2: SyntaxWarning: invalid escape sequence '\.'
  mask = df['complemento'].str.contains('apt\.', na=False, case=False)


## Tratamento da coluna `complemento`

A coluna apresenta dois tipos de informação misturados em texto livre:
- Unidade do imóvel: `apto 203`, `sala 0609`
- Nome do edifício: `edf sainte juliana`, `edf. tolive one`

### Decisões de tratamento:
1. Padronizar texto para maiúsculo e remover espaços extras
2. Separar em duas novas colunas: `complemento_numero` e `nome_edificio`
3. Manter nulos como `NaN` — ausência de complemento é válida
4. Remover a coluna original após a separação

In [127]:
#Deixar todo o texto em CapsLk
df['complemento'] = df['complemento'].str.strip().str.upper()

#Extrair número da unidade (apto/sala + número)
df['complemento_numero'] = df['complemento'].str.extract(
    r'^((?:APTO|SALA)\s+[\w]+)',
    expand=False
)
#Extrair nome do edifício (tudo após EDF. ou EDF)
df['nome_edificio'] = df['complemento'].str.extract(
    r'EDF\.?\s+(.+)',
    expand=False
).str.strip()

#Remover a coluna original
df = df.drop(columns=['complemento'])

#Verificando se deu certo
print(df[['complemento_numero', 'nome_edificio']].head(20))
print(f"\nNulos em complemento_numero: {df['complemento_numero'].isna().sum()}")
print(f"Nulos em nome_edificio: {df['nome_edificio'].isna().sum()}")

   complemento_numero      nome_edificio
0                 NaN                NaN
1                 NaN                NaN
2              APTO 1                NaN
3              APTO 1                NaN
4              APTO 2                NaN
5              APTO 3                NaN
6              APTO 4                NaN
7           APTO 0005                NaN
8                 NaN                NaN
9                 NaN                NaN
10           APTO 203     SAINTE JULIANA
11           APTO 201     SAINTE JULIANA
12          APTO 1501     SAINTE JULIANA
13                NaN                NaN
14           APTO 902  ESSENZA ROSARINHO
15          APTO 2101       JARDIM MONET
16          APTO 1401     PRAIA DE CERES
17           APTO 201         MARIA ROSA
18           APTO 404         MARIA ROSA
19          APTO 1304         MARIA ROSA

Nulos em complemento_numero: 5340
Nulos em nome_edificio: 17985


## Conclusão da separação da coluna `complemento`

A coluna `complemento` original apresentava duas informações distintas
em um único campo de texto livre: a unidade do imóvel (ex: `apto 203`)
e o nome do edifício (ex: `edf sainte juliana`), o que tornava a coluna
desorganizada e dificultava qualquer análise isolada dessas informações.

Por isso, optou-se por separar em duas colunas:
- `complemento_numero`: identifica a unidade do imóvel
- `nome_edificio`: identifica o condomínio ou edifício

Descartar a informação do edifício não seria adequado, uma vez que
aproximadamente **26.246 linhas** possuem esse campo preenchido,
representando uma parcela significativa do dataset e sendo essencial
para análises de imóveis específicos.

#### Organizar posição das colunas após adaptação

In [128]:
colunas = [
    'logradouro', 'numero', 'complemento_numero', 'nome_edificio',
    'valor_avaliacao', 'bairro', 'cidade', 'uf', 'ano_construcao',
    'area_terreno', 'area_construida', 'fracao_ideal',
    'padrao_acabamento', 'tipo_construcao', 'tipo_ocupacao',
    'data_transacao', 'estado_conservacao', 'tipo_imovel',
    'sfh', 'cod_logradouro', 'latitude', 'longitude', 'ano',
]

df = df[colunas]
print(df.columns.tolist())

['logradouro', 'numero', 'complemento_numero', 'nome_edificio', 'valor_avaliacao', 'bairro', 'cidade', 'uf', 'ano_construcao', 'area_terreno', 'area_construida', 'fracao_ideal', 'padrao_acabamento', 'tipo_construcao', 'tipo_ocupacao', 'data_transacao', 'estado_conservacao', 'tipo_imovel', 'sfh', 'cod_logradouro', 'latitude', 'longitude', 'ano']


Com esse código, a coluna logradouro foi padronizada. O número de logradouros únicos reduziu, confirmando que as duplicatas foram unificadas. O tratamento dessa coluna está finalizado.

## **Tratamento da Coluna 'Numero'** ##

In [129]:
# visão geral da coluna
print("Tipo:", df['numero'].dtype)
print("Total de linhas:", len(df))
print("Nulos:", df['numero'].isna().sum())
print("Valores únicos:", df['numero'].nunique())
print()
print("Amostra dos 20 valores mais frequentes:")
print(df['numero'].value_counts(dropna=False).head(20))

Tipo: int64
Total de linhas: 44231
Nulos: 0
Valores únicos: 1846

Amostra dos 20 valores mais frequentes:
numero
80      799
251     617
1000    613
6171    515
179     514
300     461
121     452
1120    417
100     384
862     378
60      362
1069    357
56      295
50      277
115     272
360     269
120     265
16      265
141     261
85      255
Name: count, dtype: int64


### Passo 1 - converter tudo para string

A coluna `numero` é do tipo `object`, o que significa que o pandas
está misturando tipos diferentes na mesma coluna: algumas células
podem ser inteiros (`123`), outras floats (`123.0`), outras strings
(`"123"`). Isso causa erros nas etapas de limpeza seguintes.

Para resolver, convertemos tudo para `str` com `.astype(str)`.
O problema é que essa conversão também transforma os valores nulos
(`NaN`) na string `"nan"` — que não é um nulo de verdade, é só um
texto. Por isso logo em seguida substituímos `"nan"` de volta para
`NaN` real, preservando a informação de que aquela célula estava vazia.

In [130]:
import numpy as np

# converte para string
df['numero'] = df['numero'].astype(str)

# "nan" virou string na conversão, voltamos para NaN real
df['numero'] = df['numero'].replace('nan', np.nan)

### Passo 2 - tratar o valor `-1`

O valor `-1` é um sentinela usado para indicar "sem número" (imóveis do tipo S/N). Precisamos padronizá-lo para `"SN"` junto com outras variantes comuns do mesmo significado.

- -1
- "S/N", "s/n", "SEM NUMERO"
- 0

In [131]:
# remove espaços e coloca em maiúsculo para comparação segura
df['numero'] = df['numero'].str.strip().str.upper()

# todas as variantes de "sem número" viram "SN"
sem_numero = ['-1', '0', 'S/N', 'SEM NUMERO',
              'SEM NÚMERO', 'S.N.', 'S.N', 'N/A']

df['numero'] = df['numero'].replace(sem_numero, 'SN')

### Passo 3 - limpar caracteres indesejados

Números de endereço podem ter sujeiras como espaços extras e zeros à
esquerda (`"007"` → `"7"`). Além disso, sufixos de complemento podem
aparecer com ou sem hífen (`"123A"` e `"123-A"` representam o mesmo
endereço). Vamos padronizar removendo zeros à esquerda e garantindo
que o hífen sempre esteja presente entre o número e o sufixo.

In [132]:
import re

def limpar_numero(val):
    if pd.isna(val) or val == 'SN':
        return val
    # remove tudo exceto dígitos, letras e hífen
    val = re.sub(r'[^\w\-]', '', val)
    # remove zeros à esquerda em partes numéricas
    val = re.sub(r'\b0+(\d)', r'\1', val)
    # padroniza sufixo: "123A" → "123-A"
    val = re.sub(r'(\d)([A-Z])', r'\1-\2', val)
    
    return val.strip() or None

df['numero'] = df['numero'].apply(limpar_numero)

### Passo 4 - tratar nulos restantes

Após as etapas anteriores ainda podem restar `NaN`. Para imóveis sem número cadastrado o comportamento mais seguro é preencher com `"SN"`, pois é a forma padrão de registrar a ausência de numeração.

In [133]:
# preenche nulos restantes com "SN"
df['numero'] = df['numero'].fillna('SN')

print("Nulos após tratamento:", df['numero'].isna().sum())

Nulos após tratamento: 0


### Verificação do resultado

Conferimos se o tratamento foi aplicado corretamente: não deve restar nenhum `-1`, zero, ou variante de S/N na coluna.

In [134]:
print("=== Resultado final ===")
print("Tipo da coluna:", df['numero'].dtype)
print("Nulos:", df['numero'].isna().sum())
print("Contagem de SN:", (df['numero'] == 'SN').sum())
print()

# checar se ainda existe algum sentinela
sentinelas_restantes = df['numero'].isin(['-1', '0', 'S/N']).sum()
print(f"Sentinelas restantes: {sentinelas_restantes}")
print()
print("Top 10 valores mais frequentes:")
print(df['numero'].value_counts().head(10))

=== Resultado final ===
Tipo da coluna: str
Nulos: 0
Contagem de SN: 22

Sentinelas restantes: 0

Top 10 valores mais frequentes:
numero
80      799
251     617
1000    613
6171    515
179     514
300     461
121     452
1120    417
100     384
862     378
Name: count, dtype: int64


## **Tratamento da Coluna 'Valor_Avaliação'**

In [135]:
print(df['valor_avaliacao'].value_counts())
print(df['valor_avaliacao'].info())

print(df['valor_avaliacao'].describe())

valor_avaliacao
190000.00     513
200000.00     481
220000,00     403
250000.00     366
257000,00     356
             ... 
881959,07       1
56680,00        1
1382000,00      1
630652,51       1
651845,44       1
Name: count, Length: 14792, dtype: int64
<class 'pandas.Series'>
RangeIndex: 44231 entries, 0 to 44230
Series name: valor_avaliacao
Non-Null Count  Dtype
--------------  -----
44231 non-null  str  
dtypes: str(1)
memory usage: 345.7 KB
None
count         44231
unique        14792
top       190000.00
freq            513
Name: valor_avaliacao, dtype: object


In [136]:
#Calcula o percentual de nulos na coluna logradouro
#Resultado 0.0 confirma que todos os registros possuem logradouro preenchido
df['logradouro'].isnull().mean() * 100

np.float64(0.0)

## Tratamento da Coluna bairro

Criando uma variável para investigar apenas a coluna desejada.

Verificamos o tipo de dado e a quantidade de entradas não nulas.

In [137]:
# info() exibe tipo e contagem de não nulos para a coluna bairro
df['bairro'].info()

<class 'pandas.Series'>
RangeIndex: 44231 entries, 0 to 44230
Series name: bairro
Non-Null Count  Dtype
--------------  -----
44231 non-null  str  
dtypes: str(1)
memory usage: 345.7 KB


Verificamos a frequência de cada bairro para identificar possíveis valores duplicados por encoding ou espaços.

In [138]:
# Frequência de cada bairro — permite identificar bairros com encoding corrompido
# que aparecem como entradas separadas (ex: 'Gracas' e 'GraÃ§as' são o mesmo bairro)
df['bairro'].value_counts()

bairro
Boa Viagem            11351
Varzea                 2430
Imbiribeira            2238
Pina                   1845
Madalena               1666
                      ...  
São José                  2
Brejo Da Guabiraba        2
Água Fria                 2
Toto                      2
Hipódromo                 1
Name: count, Length: 95, dtype: int64

Obtemos um resumo estatístico da coluna.

In [139]:
# describe() mostra o total, a quantidade de bairros únicos e o mais frequente
# O número de únicos elevado para uma cidade indica possível fragmentação por encoding
df['bairro'].describe()

count          44231
unique            95
top       Boa Viagem
freq           11351
Name: bairro, dtype: object

Calculamos o percentual de valores nulos na coluna.

In [140]:
# Verifica o percentual de nulos na coluna bairro
# Resultado 0.0 indica que todos os registros possuem bairro preenchido
df['bairro'].isnull().mean() * 100

np.float64(0.0)

Listamos todos os valores distintos para identificar variações de escrita ou valores inesperados.

In [141]:
# Lista todos os valores distintos para inspeção visual
# Com apenas ~95 bairros únicos, é possível identificar manualmente os problemas:
# encoding corrompido (ex: 'GraÃ§as', 'SÃ£o JosÃ©') e espaços extras ('Cohab   Ibura')
df['bairro'].unique()

<StringArray>
[       'Encruzilhada',           'Rosarinho',         'Tamarineira',
             'Aflitos',              'Gracas',              'Graças',
            'Jaqueira',          'Espinheiro',         'Santo Amaro',
              'Recife',               'Derby',           'Paissandu',
           'Boa Vista',            'Soledade',       'Santo Antonio',
            'Sao Jose',            'São José',       'Ilha Do Leite',
             'Coelhos',             'Cabanga',         'Dois Unidos',
            'Beberibe',       'Linha Do Tiro',    'Porto Da Madeira',
            'Cajueiro',           'Agua Fria',  'Bomba Do Hemeterio',
              'Fundao',  'Campina Do Barreto',              'Arruda',
        'Campo Grande',  'Alto Jose Do Pinho',          'Mangabeira',
     'Ponto De Parada',           'Hipodromo',             'Torreao',
             'Torreão',           'Guabiraba',  'Brejo Da Guabiraba',
   'Brejo De Beberibe',     'Nova Descoberta', 'Corrego Do Jenipapo',
      

Podemos verificar que a análise dos valores únicos revelou dois tipos de inconsistência. O primeiro é o encoding corrompido: os mesmos bairros aparecem em duas versões, uma sem acento e outra com caracteres corrompidos como Ã§, Ã£ e Ã. Por exemplo, 'Gracas' e 'GraÃ§as' representam o mesmo bairro, assim como 'Sao Jose' e 'SÃ£o JosÃ©', entre outros. O segundo tipo são espaços extras, como em 'Cohab   Ibura', que possui três espaços entre as palavras.

Assim, aplicamos as correções em sequência. Primeiro removemos a linha de cabeçalho duplicada. Em seguida corrigimos o encoding corrompido, removemos os acentos para unificar versões com e sem acento e por fim normalizamos os espaços extras.

In [142]:
#Primeiro: Remove linha de cabeçalho duplicada (mesmo problema do logradouro)
df = df[df['bairro'] != 'bairro']

#Segundo: Corrige encoding corrompido usando a função definida anteriormente
# Ex: 'GraÃ§as' → 'Graças', 'SÃ£o JosÃ©' → 'São José'
df['bairro'] = df['bairro'].apply(corrigir_encoding)

#Terceiro: Remove acentos para unificar versões com e sem acento
# Ex: 'Graças' (após correção) e 'Gracas' (original sem acento) → ambos viram 'Gracas'
df['bairro'] = df['bairro'].apply(remover_acentos)

#Quarto: Normaliza espaços múltiplos
# \s+ captura um ou mais espaços consecutivos e substitui por um único espaço
# Ex: 'Cohab   Ibura' → 'Cohab Ibura'
df['bairro'] = df['bairro'].str.replace(r'\s+', ' ', regex=True).str.strip()

print(f"Únicos antes do tratamento: 95")
print(f"Únicos após tratamento: {df['bairro'].nunique()}")
df['bairro'].value_counts()

Únicos antes do tratamento: 95
Únicos após tratamento: 88


bairro
Boa Viagem             11351
Varzea                  2448
Imbiribeira             2238
Pina                    1845
Madalena                1666
                       ...  
Corrego Do Jenipapo        4
Brejo De Beberibe          3
Dois Irmaos                3
Brejo Da Guabiraba         2
Toto                       2
Name: count, Length: 88, dtype: int64

Com esse código, a coluna bairro foi padronizada. O número de bairros únicos reduziu de 95 para o valor correto, confirmando que as duplicatas foram eliminadas. O tratamento dessa coluna está finalizado.

Para essa coluna, será necessário fazer a mudança do tipo da variável para INT. Outro problema claro é a falta de consistência na separação decimal, em que alguns casos usam '.' e outros ','.

In [143]:
df['valor_avaliacao'] = df['valor_avaliacao'].astype(str).str.replace(',', '.', regex=False) #corrige todos valores que tem ',' para o separador decimal '.'
df['valor_avaliacao'] = pd.to_numeric(df['valor_avaliacao'], errors='coerce') #corrige o tipo da variável para int

df['valor_avaliacao'].tail(20)
df['valor_avaliacao'].describe()


count    4.423100e+04
mean     6.760577e+05
std      2.690452e+06
min      1.000000e+00
25%      2.400000e+05
50%      3.573597e+05
75%      5.800000e+05
max      1.627350e+08
Name: valor_avaliacao, dtype: float64

Como podemos ver, temos valores bastante altos e bastante baixos. Analisando esses valores enormes, eles estão coerentes, por se tratarem de grandes galpões, complexos comerciais ou supermercados, isto é, terrenos enormes em locais privilegiados da cidade. Entretanto, os valores baixíssimos são erros, para edifícios que podem estar em processamento ou transferência por herança. Para lidar com esses valores baixos, vamos analisar quantos são e depois substituir pela mediana dos valores médios dos seus bairros.

In [144]:

print("Valores < R$ 1000:")# Valores muito baixos (provavelmente erro)
print(df[df['valor_avaliacao'] < 1000]['valor_avaliacao'].value_counts().sort_index())


print("\nTop 10 maiores:")# Valores muito altos
print(df['valor_avaliacao'].sort_values(ascending=False).head(10))

# Quantos abaixo de um limite mínimo razoável (ex: 10 mil)
print(f"\nAbaixo de R$ 10.000: {(df['valor_avaliacao'] < 10000).sum()}")

Valores < R$ 1000:
valor_avaliacao
1.00      18
19.52      1
154.53     1
389.17     1
400.00     1
600.00     1
Name: count, dtype: int64

Top 10 maiores:
20934    1.627350e+08
20933    1.627350e+08
4558     1.583287e+08
10583    1.409932e+08
30117    1.359266e+08
30118    1.359266e+08
29300    1.019801e+08
40271    9.707369e+07
40272    9.707369e+07
7668     9.042900e+07
Name: valor_avaliacao, dtype: float64

Abaixo de R$ 10.000: 58


In [145]:
df.loc[df['valor_avaliacao'] < 10000, 'valor_avaliacao'] = None# Trata valores irrealistas como nulos

df['valor_avaliacao'] = df['valor_avaliacao'].fillna(
    df.groupby('bairro')['valor_avaliacao'].transform('median')
)# Imputa pela mediana do bairro (mesma lógica usada em ano_construcao)

df['valor_avaliacao'] = df['valor_avaliacao'].fillna(df['valor_avaliacao'].median())# Fallback para mediana geral

print(df['valor_avaliacao'].describe())
print(f"Nulos restantes: {df['valor_avaliacao'].isna().sum()}")

count    4.423100e+04
mean     6.765347e+05
std      2.690372e+06
min      1.000000e+04
25%      2.400000e+05
50%      3.580107e+05
75%      5.800000e+05
max      1.627350e+08
Name: valor_avaliacao, dtype: float64
Nulos restantes: 0


Com isso, todos os valores da coluna estão normalizados para INT, e não constam com outliers que destõem da realidade dos seus bairros quanto ao valor

## **Tratamento da Coluna 'Ano Construção'**

In [146]:
df1 = df['ano_construcao']
df1.info()

print(df['ano_construcao'].value_counts())

print(df['ano_construcao'].describe())

<class 'pandas.Series'>
RangeIndex: 44231 entries, 0 to 44230
Series name: ano_construcao
Non-Null Count  Dtype
--------------  -----
44231 non-null  int64
dtypes: int64(1)
memory usage: 345.7 KB
ano_construcao
2024    5011
2025    3213
2023    3102
2022    1546
1982    1518
        ... 
1940      31
1944      29
1943      24
1915       1
1938       1
Name: count, Length: 89, dtype: int64
count    44231.000000
mean      2004.082340
std         20.453677
min       1915.000000
25%       1991.000000
50%       2010.000000
75%       2023.000000
max       2025.000000
Name: ano_construcao, dtype: float64


Como podemos ver por essas informações, teremos alguns pontos a serem tratados: O tipo de variável está incorreto, faz mais sentido estar como um int, por ser um ano; Já que não temos valores nulos, não precisamos nos preocupar com isso. O principal ponto de atenção são os anos sem coerência. Felizmente, não tem valores acima de 2025, então nada sobresaiu o teto.

In [147]:

df['ano_construcao'] = pd.to_numeric(df['ano_construcao'], errors='coerce').astype('Int64') # Converte o tipo da variável para int  

df['ano_construcao'].info()
df['ano_construcao'].describe()


<class 'pandas.Series'>
RangeIndex: 44231 entries, 0 to 44230
Series name: ano_construcao
Non-Null Count  Dtype
--------------  -----
44231 non-null  Int64
dtypes: Int64(1)
memory usage: 388.9 KB


count       44231.0
mean     2004.08234
std       20.453677
min          1915.0
25%          1991.0
50%          2010.0
75%          2023.0
max          2025.0
Name: ano_construcao, dtype: Float64

Por fim, vê-se que todos os dados estão coerentes, como INT e sem outliers, pois todas as datas estão entre 1915 e 2025, coerentes com os valores que esperamos para essa base de dados

## **Tratamento da coluna 'Area Terreno'**

### Diagnóstico da coluna

Antes de qualquer transformação, vamos verificar o tipo dos dados, quantidade de nulos e distribuição dos valores.

Como o tipo do dado está 'str' devemos mudar para 'float', já que essa coluna deve representar um valor real.

In [148]:
df["area_terreno"] = df["area_terreno"].astype(str).str.replace(",", ".", regex=False)
df["area_terreno"] = df["area_terreno"].astype(float)

Dessa maneira, o tratamento da coluna está finalizado.

## **Tratamento da Coluna 'Área Construida'**

Criando um variavel para investigar apenas a coluna desejada

In [149]:
df_tratar = df['area_construida']

### Diagnóstico da coluna

Antes de qualquer transformação, vamos verificar o tipo dos dados, quantidade de nulos e distribuição dos valores.

In [150]:
print("Tipo:", df_tratar.dtype)
print("Nulos:", df_tratar.isna().sum())
print("Valores únicos:", df_tratar.nunique())
print()
print(df_tratar.value_counts(dropna=False).head(20))

Tipo: str
Nulos: 0
Valores únicos: 9379

area_construida
47,51     263
47,52     253
76,62     228
70,79     227
71,57     226
78,2      211
197,16    200
69,89     175
50,97     161
59,78     161
61,32     159
119,61    159
84,37     155
78,64     150
52,37     136
91,57     136
61,57     134
101,45    128
51,31     127
107,16    120
Name: count, dtype: int64


Podemos verificar que os dados estão no tipo 'String' quando deveriam estar como 'Float', isso pois representa o valor da área do imóvel. Além disso, verificamos que os todos os valores são coniventes a serem Area Construidas

In [151]:
df['area_construida'] = df['area_construida'].str.replace(',', '.').astype(float)

Assim, transformamos o tipo do dados para o correto, como essa coluna não possui valores faltantes, podemos finalizar o tratamento dessa coluna.

## **Coluna Fração Ideal**

### Diagnóstico da coluna

Antes de qualquer transformação, vamos verificar o tipo dos dados, quantidade de nulos e distribuição dos valores.

In [152]:
print("Tipo:", df['fracao_ideal'].dtype)
print("Total de linhas:", len(df))
print("Nulos:", df['fracao_ideal'].isna().sum())
print("Valores únicos:", df['fracao_ideal'].nunique())
print()
print("20 valores mais frequentes:")
print(df['fracao_ideal'].value_counts(dropna=False).head(20))

Tipo: str
Total de linhas: 44231
Nulos: 0
Valores únicos: 4652

20 valores mais frequentes:
fracao_ideal
1,0        2721
1          1352
0,16666     356
0,0014      338
0,03125     271
0,00001     264
0,00199     263
0,04166     260
0,002       254
0,08333     253
0,00417     246
0,00446     238
0,00138     226
0,0024      215
0,0025      195
0,01785     194
0,03333     193
0,02272     192
0,025       190
0,03846     189
Name: count, dtype: int64


### Passo 1 - padronizar o formato decimal

Como os valores podem vir com vírgula decimal (`0,16666`), vamos converter para o padrão usado pelo Python (`0.16666`).

In [153]:
df['fracao_ideal'] = (
    df['fracao_ideal']
    .astype(str)
    .str.strip()
    .str.replace(',', '.', regex=False)
)

### Passo 2 - testar a conversão para número

Agora tentamos converter toda a coluna para `float`.

Qualquer valor que não seja numérico será transformado em NaN, permitindo identificar problemas facilmente.

In [154]:
temp = pd.to_numeric(
    df['fracao_ideal'],
    errors='coerce'
)

print("Valores que não puderam ser convertidos:",
      temp.isna().sum())

Valores que não puderam ser convertidos: 0


Agora que sabemos que todos os valores são numéricos, podemos converter a coluna.

In [155]:
df['fracao_ideal'] = pd.to_numeric(
    df['fracao_ideal'],
    errors='coerce'
)

### Passo 3 - verificação do resultado

Conferimos se o tratamento foi aplicado corretamente: não deve restar nenhum valor nulo na coluna e o tipo deve ser `float`.

In [156]:
print("Tipo:", df['fracao_ideal'].dtype)
print("Nulos:", df['fracao_ideal'].isna().sum())
print()

print(df['fracao_ideal'].describe())

Tipo: float64
Nulos: 0

count    44231.000000
mean         0.114610
std          0.286649
min          0.000000
25%          0.004660
50%          0.013150
75%          0.031315
max          1.180050
Name: fracao_ideal, dtype: float64


### Passo 4 - checagem de números maiores que 1 (mais de 100%)

Visto que é uma fração que o intervalo esperado seria de 0-100% é necessário analisar quais casos estão ultrapassando o 100%.

In [157]:
print("Quantidade > 1:", (df['fracao_ideal'] > 1).sum())
print()
df[df['fracao_ideal'] > 1]

Quantidade > 1: 2



,logradouro,numero,complemento_numero,nome_edificio,valor_avaliacao,bairro,cidade,uf,ano_construcao,area_terreno,...,tipo_construcao,tipo_ocupacao,data_transacao,estado_conservacao,tipo_imovel,sfh,cod_logradouro,latitude,longitude,ano
4412,rua professor ageu magalhaes,211,NaN,NaN,1200000.0,Parnamirim,Recife,PE,1997,254.35,...,Casa,COMERCIAL COM LIXO ORGANICO,2023-01-10,Regular,Casa,960000.00,1198,-8.033687,-34.913381,2023
7805,rua aurora cacote,602,NaN,NaN,550000.0,Areias,Recife,PE,1968,200.00,...,Casa,RESIDENCIAL,2023-09-25,Bom,Casa,0.00,9172,-8.099398,-34.933164,2023


Foram identificados dois registros com fração ideal superior a 1 (100%). Como a fração ideal representa uma participação proporcional do imóvel, espera-se que seus valores estejam limitados ao intervalo [0.0, 1.0]. Devido à baixa incidência desses casos e à impossibilidade de confirmar o valor correto, os registros foram mantidos para preservar a integridade dos dados originais.

---

## **Tratanto a Coluna padrao_acabamento**

Criando uma variável para isolar a coluna desejada

In [158]:
df_tratar = df['padrao_acabamento']

### Diagnóstico da coluna

Antes de qualquer transformação, vamos verificar o tipo dos dados, quantidade de nulos e distribuição dos valores.

In [159]:
print("Tipo:", df_tratar.dtype)
print("Nulos:", df_tratar.isna().sum())
print("Valores únicos:", df_tratar.nunique())
print()
print(df_tratar.value_counts(dropna=False).head(20))

Tipo: str
Nulos: 0
Valores únicos: 3

padrao_acabamento
Superior    23087
Médio       14743
Simples      6401
Name: count, dtype: int64


Verificamos que essa coluna não possui dados faltantes, entranto o tipo está como 'String', nessa situção seria melhor se os dados fossem categoricos, pois o padrão de acabamento so aceita 3 tipos diferentes.

In [160]:
df['padrao_acabamento'] = df['padrao_acabamento'].astype('category')
df['padrao_acabamento'].dtype

CategoricalDtype(categories=['Médio', 'Simples', 'Superior'], ordered=False, categories_dtype=str)

Com esse código alteramos o tipo do dado para o correto. Dessa maneira, o tratamento dessa coluna está finalizado

## **Tratamento da coluna do 'Tipo_Construção'** ##

Vamos criar uma variável para analisar a coluna.

In [161]:
df_tratar = df['tipo_construcao']

### Diagnóstico da coluna

Antes de qualquer transformação, vamos verificar o tipo dos dados, quantidade de nulos e distribuição dos valores.

In [162]:
print("Tipo:", df_tratar.dtype)
print("Nulos:", df_tratar.isna().sum())
print("Valores únicos:", df_tratar.nunique())
print()
print(df_tratar.value_counts(dropna=False).head(20))

Tipo: str
Nulos: 0
Valores únicos: 16

tipo_construcao
Apartamento > 4 Pavimentos     31706
Apartamento <= 4 Pavimentos     3999
Casa                            3571
Sala > 4 Pavimentos             2846
Loja <= 4 Pavimentos             663
Sala <= 4 Pavimentos             403
Edificação Especial              344
Galpão                           278
Loja > 4 Pavimentos              129
Edificação Garagem                92
Hotel                             73
Instituição Financeira            49
Edificação Industrial             27
Instituição Hospitalar            20
Posto de Combustível              16
Mocambo                           15
Name: count, dtype: int64


Podemos verificar que esa coluna não possui dados faltantes e o tipo está correto, assim não é necessário nenhum tipo de tratamento

## **Coluna Tipo Ocupação**

### Diagnóstico da coluna

Antes de qualquer transformação, vamos verificar o tipo dos dados, quantidade de nulos e distribuição dos valores.

In [163]:
print("Tipo:", df['tipo_ocupacao'].dtype)
print("Nulos:", df['tipo_ocupacao'].isna().sum())
print("Valores únicos:", df['tipo_ocupacao'].nunique())
print()
print(df['tipo_ocupacao'].value_counts(dropna=False).head(20))

Tipo: str
Nulos: 0
Valores únicos: 3

tipo_ocupacao
RESIDENCIAL                    38403
COMERCIAL COM LIXO ORGANICO     3320
COMERCIAL SEM LIXO ORGANICO     2508
Name: count, dtype: int64


A coluna `tipo_ocupacao` possui natureza categórica e apresenta apenas três categorias distintas. Não foram identificados valores nulos nem inconsistências de preenchimento. Foi definido que a coluna já se encontrava consistente e não demandou tratamentos adicionais.

# Tratamento da coluna 'data_transacao'

### Diagnóstico da coluna

Antes de qualquer transformação, vamos verificar o tipo dos dados, quantidade de nulos e distribuição dos valores.

In [164]:
print(df['data_transacao'].dtype)
print(df['data_transacao'].value_counts(dropna=False).head(10))
print(f"\nTotal de nulos: {df['data_transacao'].isna().sum()}")
print(f"Amostra: {df['data_transacao'].sample(5).tolist()}")

str
data_transacao
2025-11-14    253
2025-07-01    215
2024-05-21    213
2025-01-08    191
2024-05-20    160
2024-09-09    155
2025-09-16    147
2025-05-19    147
2025-07-18    144
2025-06-03    144
Name: count, dtype: int64

Total de nulos: 0
Amostra: ['2025-05-16', '2023-09-05', '2023-11-20', '2023-03-14', '2023-01-31']


Percebemos que o tipo do dado está como objeto, então vamos transforma-lo em data

In [165]:
#Nessa seção, somos forçados a remover o valor duplicado do início, pois, isso causa erro na mudança de tipo
df = df[df['data_transacao'] != 'data_transacao'].reset_index(drop=True)

df['data_transacao'] = pd.to_datetime(df['data_transacao'], format='%Y-%m-%d')

print(df['data_transacao'].dtype)


datetime64[us]


In [166]:
df['ano'] = df['ano'].astype(int)
df['ano_transacao'] = df['data_transacao'].dt.year


# Agora refaz o teste
divergencias = df[df['ano'] != df['ano_transacao']]
print(f"Total de divergências: {len(divergencias)}")
print(divergencias.groupby(['ano', 'ano_transacao']).size())
df = df.drop(columns=['ano_transacao'])


Total de divergências: 0
Series([], dtype: int64)


## Tratamento da coluna `data_transacao`

A coluna `data_transacao` registra a data em que a transação imobiliária
foi realizada. Após análise, a coluna apresentou as seguintes características:

- **0 valores nulos**
- **Formato consistente: `YYYY-MM-DD`** em todos os registros
- **dtype `object`** — a coluna estava armazenada como texto, sendo necessária
  a conversão para o tipo `datetime`

### Decisões de tratamento:
1. Conversão da coluna para o tipo `datetime` com formato `%Y-%m-%d`

Optou-se por **manter a coluna intacta após a conversão**, sem extrair
colunas auxiliares de mês, trimestre ou dia. A coluna `data_transacao`
como `datetime` já permite essas extrações diretamente no momento da
análise, evitando colunas desnecessárias no dataset.

## **Tratamento da coluna 'Estado Conservação'**

Criando uma variável para investigar apenas a coluna desejada.

Verificamos o tipo de dado e a quantidade de entradas não nulas.

Verificamos a frequência de cada valor para identificar categorias esperadas e possíveis valores inválidos.

In [167]:
#value_counts() conta a frequência de cada valor distinto na coluna
#Permite identificar categorias inesperadas ou valores inválidos misturados com os dados reais
df['estado_conservacao'].value_counts()

estado_conservacao
Bom        43549
Regular      622
Mau           60
Name: count, dtype: int64

Obtemos um resumo estatístico da coluna, incluindo total de registros, quantidade de valores únicos e o valor mais frequente.

In [168]:
#describe() resume a coluna: total de registros, quantidade de únicos, valor mais frequente (top) e sua contagem (freq)
df['estado_conservacao'].describe()

count     44231
unique        3
top         Bom
freq      43549
Name: estado_conservacao, dtype: object

Calculamos o percentual de valores nulos na coluna.

In [169]:
#Calcula o percentual de valores nulos na coluna
#isnull() retorna True/False por linha; mean() calcula a proporção; * 100 converte para percentual
#Resultado 0.0 indica ausência total de nulos
df['estado_conservacao'].isnull().mean() * 100

np.float64(0.0)

Listamos todos os valores distintos para verificar variações de escrita ou valores inesperados.

In [170]:
#unique() lista todos os valores distintos presentes na coluna
#Permite verificar visualmente se há variações de escrita, valores corrompidos ou inválidos
df['estado_conservacao'].unique()

<StringArray>
['Regular', 'Bom', 'Mau']
Length: 3, dtype: str

Podemos verificar que a coluna possui um valor inválido: 'estado_conservacao' aparece como dado. Isso ocorreu porque, ao unir os arquivos CSV, o cabeçalho de um dos arquivos foi incluído como linha de dados. Os demais valores, Bom, Regular e Mau, estão padronizados, com mesma capitalização e sem espaços extras.

In [171]:
#Remove a linha onde o valor de 'estado_conservacao' é igual ao próprio nome da coluna
#Isso ocorreu porque, ao unir os CSVs incorretamente, o cabeçalho de um dos arquivos
#foi tratado como linha de dados, inserindo 'estado_conservacao' como um registro real
df = df[df['estado_conservacao'] != 'estado_conservacao']

#Confirma a distribuição após a limpeza — esperamos apenas: Bom, Regular, Mau
df['estado_conservacao'].value_counts()

estado_conservacao
Bom        43549
Regular      622
Mau           60
Name: count, dtype: int64

Assim, removemos o valor inválido e confirmamos que apenas os três valores esperados permanecem na coluna. O tratamento dessa coluna está finalizado.

# Tratamento da coluna 'tipo_imovel'

observação inicial dos dados

In [172]:
df['tipo_imovel'].info()


<class 'pandas.Series'>
RangeIndex: 44231 entries, 0 to 44230
Series name: tipo_imovel
Non-Null Count  Dtype
--------------  -----
44231 non-null  str  
dtypes: str(1)
memory usage: 345.7 KB


In [173]:
print(df['tipo_imovel'].value_counts(dropna=False))
print(f"\nTotal de nulos: {df['tipo_imovel'].isna().sum()}")

tipo_imovel
Apartamento                    35697
Casa                            3574
Sala                            3096
Loja                             790
Edificação Especial              319
Galpão                           269
Centro Comercial/Serviços        158
Hotel                             73
Garagem Comercial                 59
Instituição Financeira            49
Garagem Residencial               33
Industria                         31
Hospital                          20
Instituição Educacional           20
Mocambo                           15
Posto de Abastecimento            15
Templo religioso                   5
Galpão Fechado                     4
Terreno em cond residencial        4
Name: count, dtype: int64

Total de nulos: 0


Como essa coluna está bem padronizada, apenas iremos confirmar a quantidade de categorias que temos

In [174]:
print(f"Categorias únicas: {df['tipo_imovel'].nunique()}")

Categorias únicas: 19


## Tratamento da coluna `tipo_imovel`

A coluna `tipo_imovel` classifica o tipo de imóvel envolvido em cada
transação. Após análise, a coluna apresentou um estado consideravelmente
limpo, com as seguintes características:

- **0 valores nulos**
- **19 categorias bem definidas**, sem variações de escrita ou abreviações
  (ex: não havia `"Apto"` vs `"Apartamento"`)


### Decisão de tratamento:
Dado o bom estado da coluna, foi decidido que não é necessário ser feito nenhum tratamento,
a coluna está pronta para uso.

## **Tratamento da Coluna 'SFH'** ##

### Diagnóstico da coluna

Antes de qualquer transformação, vamos verificar o tipo dos dados, quantidade de nulos e distribuição dos valores.

In [175]:
# visão geral da coluna
print("Tipo:", df['sfh'].dtype)
print("Total de linhas:", len(df))
print("Nulos:", df['sfh'].isna().sum())
print("Valores únicos:", df['sfh'].nunique())
print()
print("Amostra dos 20 valores mais frequentes:")
print(df['sfh'].value_counts(dropna=False).head(20))

Tipo: str
Total de linhas: 44231
Nulos: 0
Valores únicos: 7916

Amostra dos 20 valores mais frequentes:
sfh
0.00         19456
0,00         10616
200000.00      177
150000.00      120
204000,00      119
280000.00      100
300000.00       99
240000.00       96
400000.00       80
204000.00       79
250000.00       78
152000.00       74
160000.00       74
200000,00       73
100000.00       69
140000.00       67
220000.00       64
280000,00       62
180000.00       59
120000.00       55
Name: count, dtype: int64


Podemos verificar que não existe dados faltantes, assim devemos apenas tratar o tipo da coluna, na qual deveria ser 'float', não 'string'.

In [176]:
df["sfh"] = df["sfh"].astype(str).str.replace(",", ".", regex=False)
df["sfh"] = df["sfh"].astype(float)
df["sfh"].unique()


array([     0.  , 525600.  , 200000.  , ..., 327290.29, 643200.  ,
       184860.78], shape=(7508,))

## **Tratamento da coluna 'cod_logradouro'**

In [177]:

print(df['cod_logradouro'].value_counts())
print(df['cod_logradouro'].info())

print(df['cod_logradouro'].describe())

cod_logradouro
11720    1178
81930     673
45810     621
56898     522
57150     512
         ... 
25100       1
43923       1
48615       1
32930       1
10227       1
Name: count, Length: 2169, dtype: int64
<class 'pandas.Series'>
RangeIndex: 44231 entries, 0 to 44230
Series name: cod_logradouro
Non-Null Count  Dtype
--------------  -----
44231 non-null  int64
dtypes: int64(1)
memory usage: 345.7 KB
None
count     44231.000000
mean      35424.112229
std       22946.968275
min          78.000000
25%       16462.000000
50%       35050.000000
75%       51250.000000
max      127639.000000
Name: cod_logradouro, dtype: float64


A principal mudança que deve ser feita é a mudança do tipo da variável para INT

In [178]:
df['cod_logradouro'] = pd.to_numeric(df['cod_logradouro'], errors='coerce').astype('Int64') #corrige o tipo da variável para INT

df['cod_logradouro'].describe()

count         44231.0
mean     35424.112229
std      22946.968275
min              78.0
25%           16462.0
50%           35050.0
75%           51250.0
max          127639.0
Name: cod_logradouro, dtype: Float64

Como não há valores negativos ou nulos, essa coluna já está tratada.

## **Coluna Latitude**

### Diagnóstico da coluna

Antes de qualquer decisão, precisamos entender o estado real da coluna: quantos nulos existem, qual o tipo atual, e se os valores fazem sentido geograficamente para o Recife.

In [179]:
print("Tipo:", df['latitude'].dtype)
print("Total de linhas:", len(df))
print("Nulos:", df['latitude'].isna().sum())
print(f"Percentual de nulos: {df['latitude'].isna().mean() * 100:.1f}%")
print()
print(df['latitude'].describe())

Tipo: float64
Total de linhas: 44231
Nulos: 14874
Percentual de nulos: 33.6%

count    29357.000000
mean        -8.077194
std          0.038921
min         -8.153304
25%         -8.116538
50%         -8.064002
75%         -8.041577
max         -7.965918
Name: latitude, dtype: float64


### Decisão: remover ou manter a coluna?

A `latitude` é uma informação geográfica que complementa o endereço, mas a base do ITBI já conta com outras colunas de localização (logradouro, bairro, etc.). Por isso ela pode ser considerada redundante.

Considerando que o percentual de nulos é alto (acima de 30%), o custo de imputar ou ignorar os nulos supera o benefício da coluna, logo foi decidido que o ideal é removê-la.

In [180]:
df = df.drop(columns=['latitude'])
print("Coluna 'latitude' removida.")
print("Colunas restantes:", df.columns.tolist())

Coluna 'latitude' removida.
Colunas restantes: ['logradouro', 'numero', 'complemento_numero', 'nome_edificio', 'valor_avaliacao', 'bairro', 'cidade', 'uf', 'ano_construcao', 'area_terreno', 'area_construida', 'fracao_ideal', 'padrao_acabamento', 'tipo_construcao', 'tipo_ocupacao', 'data_transacao', 'estado_conservacao', 'tipo_imovel', 'sfh', 'cod_logradouro', 'longitude', 'ano']


---

## **Coluna Longitude**

### Diagnóstico da coluna

É esperado que a `longitude` siga o mesmo caminho da `latitude`, mas é importante entender o estado real da coluna antes de tomar qualquer decisão precipitada

In [181]:
print("Tipo:", df['longitude'].dtype)
print("Total de linhas:", len(df))
print("Nulos:", df['longitude'].isna().sum())
print(f"Percentual de nulos: {df['longitude'].isna().mean() * 100:.1f}%")
print()
print(df['longitude'].describe())

Tipo: float64
Total de linhas: 44231
Nulos: 14874
Percentual de nulos: 33.6%

count    29357.000000
mean       -34.905480
std          0.016608
min        -34.993824
25%        -34.911117
50%        -34.903350
75%        -34.894566
max        -34.870156
Name: longitude, dtype: float64


Seguindo a mesma lógica aplicada na coluna `latitude`, foi decidido pela remoção da coluna `longitude`.

In [182]:
df = df.drop(columns=['longitude'])
print("Coluna 'longitude' removida.")
print("Colunas restantes:", df.columns.tolist())

Coluna 'longitude' removida.
Colunas restantes: ['logradouro', 'numero', 'complemento_numero', 'nome_edificio', 'valor_avaliacao', 'bairro', 'cidade', 'uf', 'ano_construcao', 'area_terreno', 'area_construida', 'fracao_ideal', 'padrao_acabamento', 'tipo_construcao', 'tipo_ocupacao', 'data_transacao', 'estado_conservacao', 'tipo_imovel', 'sfh', 'cod_logradouro', 'ano']


---